# STIC model validation on California sites

Validation of the model using field data from the ameriflux database and landsat acquisitions

In [ ]:
import os

os.environ["NUMBA_DISABLE_JIT"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os
import re
import traceback
import warnings
import zipfile
from pathlib import Path

import numpy as np
import numpy.typing as npt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.autonotebook import tqdm

from sweat.common import utils
from sweat.common.flux import CST_SB, compute_et_from_le
from sweat.stic.constant import KELVIN_CST
from sweat.stic.convert import (
    convert_kelvin_to_celsius,
    convert_to_local_time,
    convert_to_rh,
)
from sweat.stic.models.functions import _tetens
from sweat.stic.runner import run_batch_init_model, run_batch_model

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

## Functions

In [ ]:
# Functions for handling EC data


def get_code(name: str) -> str:
    """
    Get code site from the name
    """
    m = re.match(r"(US-.*)[_,-].*", name)
    name = m.group(1) if m else f"US-{name}"
    if len(name) > 6:
        name = name[0:6]
    return name[:4].upper() + name[4:].lower()


def get_ec_data_filename(name: str, root: str) -> str:
    """
    Get EC data path for a site
    """
    code = get_code(name)
    files = [
        f
        for f in os.listdir(root)
        if re.match(r"AMF_US-(.*)_BASE_HH_.*\.csv", f)
    ]
    for file in files:
        if code.lower() in file.lower():
            return os.path.join(root, file)
    msg = f"File {file} for ground truth data not found"
    raise OSError(msg)

In [ ]:
# Functions for handling Ameriflux EC data


def get_ameriflux_archive(name: str, path: str) -> str:
    """
    Get EC data path for a site
    """
    files = [
        f
        for f in os.listdir(path)
        if re.match(rf"AMF_{name}_BASE-BADM_\d+-\d+.zip", f)
    ]
    if len(files) > 0:
        # Return first file
        return os.path.join(path, files[0])
    msg = f"File for site {name} not found"
    raise OSError(msg)


def get_best_column(df: pd.DataFrame, suffix: str):
    """
    Get best column to process
    """
    shift = suffix.count("_")
    if df.columns.str.startswith(f"{suffix}_PI_F").any():
        return min(
            (c for c in df.columns if c.startswith(f"{suffix}_PI_F")),
            key=lambda c: (
                (0,)
                if c == f"{suffix}_PI_F"
                else (1, *c.split("_")[3 + shift :])
            ),
        )
    if df.columns.str.startswith(f"{suffix}_PI").any():
        return min(
            (
                c
                for c in df.columns
                if c.startswith(f"{suffix}_PI")
                and not c.startswith(f"{suffix}_PI_F")
            ),
            key=lambda c: (
                (0,) if c == f"{suffix}_PI" else (1, *c.split("_")[2 + shift :])
            ),
        )
    if df.columns.str.startswith(f"{suffix}_").any():
        return min(
            (c for c in df.columns if c.startswith(suffix)),
            key=lambda c: (
                (0,) if c == suffix else (1, *c.split("_")[1 + shift :])
            ),
        )
    if suffix in df.columns:
        return suffix
    return None


def process_ameriflux_column(
    df: pd.DataFrame, col: str, name: str | None = None
) -> pd.DataFrame:
    """
    Process a column of Ameriflux data
    """
    # LE
    best = get_best_column(df, col)
    if best is None:
        best = col
        df[col] = np.nan
    if name is None:
        name = col.lower()
    return df.rename(columns={best: name})


def read_ameriflux_data(name: str) -> pd.DataFrame:
    """
    Read EC data file for a site
    """
    # Read
    df = pd.read_csv(
        name,
        sep=",",
        encoding="ascii",
        skiprows=2,
        usecols=(
            lambda col: (
                col
                in ["TIMESTAMP_START", "TIMESTAMP_END", "G", "TA", "LE", "H"]
                or col.startswith(
                    (
                        "TA_",
                        "RH",
                        "G_PI_F",
                        "H_PI_F",
                        "LE_PI_F",
                        "NETRAD",
                        "SW_IN",
                        "SW_OUT",
                        "LW_IN",
                        "LW_OUT",
                        "TS",
                    )
                )
            )
        ),
        dtype=str,
    )
    df["TIMESTAMP_START"] = pd.to_datetime(df["TIMESTAMP_START"])
    df["TIMESTAMP_END"] = pd.to_datetime(df["TIMESTAMP_END"])
    df[df.columns.difference(["TIMESTAMP_START", "TIMESTAMP_END"])] = (
        pd.to_numeric(
            df[
                df.columns.difference(["TIMESTAMP_START", "TIMESTAMP_END"])
            ].stack(),
            errors="coerce",
        ).unstack()
    )
    # Rename columns
    df = df.rename(columns={"TIMESTAMP_START": "start", "TIMESTAMP_END": "end"})
    # print(df.columns)
    # LE
    df = process_ameriflux_column(df, "LE")
    # H
    df = process_ameriflux_column(df, "H")
    # RN
    df = process_ameriflux_column(df, "NETRAD", "rn")
    # SW IN
    df = process_ameriflux_column(df, "SW_IN")
    # SW OUT
    df = process_ameriflux_column(df, "SW_OUT")
    # LW IN
    df = process_ameriflux_column(df, "LW_IN")
    # LW OUT
    df = process_ameriflux_column(df, "LW_OUT")
    # G
    df = process_ameriflux_column(df, "G")
    # TA
    df = process_ameriflux_column(df, "TA")
    # TS
    df = process_ameriflux_column(df, "TS")
    # RH
    df = process_ameriflux_column(df, "RH")
    # Keep required columns
    df = df[
        [
            "start",
            "end",
            "le",
            "h",
            "g",
            "rn",
            "sw_out",
            "sw_in",
            "lw_out",
            "lw_in",
            "ta",
            "ts",
            "rh",
        ]
    ]
    # Exclude nodata
    value_cols = df.columns.difference(["start", "end"])
    # replace -9999 with NaN only in value columns
    df[value_cols] = df[value_cols].replace(-9999.0, np.nan)
    # drop rows where all value column is NaN
    df = df.dropna(subset=value_cols, how="all")
    with warnings.catch_warnings(action="ignore"):
        # Compute EF
        df["ef"] = df["le"] / (df["h"] + df["le"])
        # Compute ET
        df["et"] = df.apply(lambda x: compute_et_from_le(x["le"]), axis=1)
        # Compute solar net radiation
        df["srn"] = df["sw_in"] - df["sw_out"]
        # Compute dewpoint temperature
        eastar = 6.13753 * np.exp((17.27 * df["ta"]) / (df["ta"] + 237.3))
        ea = (df["rh"] / 100) * (eastar)
        df["td"] = 237.3 * np.log(ea / 6.13753) / (17.27 - np.log(ea / 6.13753))
        # Compute Rn - G
        df["rn-g"] = df["rn"] - df["g"]
        # Compute LE closed
        df["le_closed"] = df["rn"] - df["g"] - df["h"]
    # Keep columns
    return df[
        [
            "start",
            "end",
            "le",
            "h",
            "ef",
            "g",
            "rn",
            "srn",
            "sw_out",
            "sw_in",
            "lw_out",
            "lw_in",
            "ta",
            "ts",
            "td",
            "rh",
            "et",
            "rn-g",
            "le_closed",
        ]
    ]


def get_ameriflux_data(df: pd.DataFrame, path: str):
    """
    Get EC data at acquisition dates
    """
    # Get sites
    sites = list(df["name"].unique())
    # Loop for each site
    datasets = []
    ec_datasets = []
    errors = []
    for site in tqdm(sites, desc="Processing sites", unit="site"):
        try:
            # Select data for a site
            selected_df = df[df["name"] == site].reset_index(drop=True)
            # Code du site
            code = get_code(site)
            # Read data
            archive = get_ameriflux_archive(code, path)
            with zipfile.ZipFile(archive, "r") as zip_ref:
                filename = f"{Path(archive).stem}.csv".replace("-BADM", "_HH")
                with zip_ref.open(filename) as file:
                    ec_df = read_ameriflux_data(file)
                    ec_df = ec_df.rename(
                        columns={
                            v: f"am_{v}"
                            for v in ec_df.columns
                            if v not in ["start", "end"]
                        }
                    )
            # Search indexes
            intervals = pd.IntervalIndex.from_arrays(
                ec_df["start"].dt.tz_localize(None),
                ec_df["end"].dt.tz_localize(None),
                closed="both",
            )
            intervals = intervals.astype("interval[datetime64[ns]]")
            matches, not_matches = intervals.get_indexer_non_unique(
                selected_df["date"].dt.tz_localize(None)
            )
            # Remove no matches
            matches = matches[matches != -1]
            datasets.append(
                selected_df.drop(not_matches).reset_index(drop=True)
            )
            ec_datasets.append(
                pd.concat(
                    [
                        selected_df[["name", "date"]]
                        .drop(not_matches)
                        .reset_index(drop=True),
                        ec_df.iloc[matches]
                        .drop(columns=["start", "end"])
                        .reset_index(drop=True),
                    ],
                    axis=1,
                )
            )
            tqdm.write(
                f"Site {site} processed with matches = {len(matches)} and not matches = {len(not_matches)}"
            )
        except Exception as e:  # noqa
            error_info = {
                "site": site,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            errors.append(error_info)
            tqdm.write(f"ERROR for site {site}: {e}")
    if len(errors) > 0:
        print("\n=== ERRORS ENCOUNTERED ===")
        for error in errors:
            print(f"\nSite: {error['site']}")
            print(f"Error: {error['error']}")
            print(error["traceback"])
    else:
        print("\nAll sites processed successfully!")

    return pd.concat(datasets), pd.concat(ec_datasets)

In [ ]:
# Functions for handling Fluxnet EC data


def get_fluxnet_archive(name: str, path: str) -> str:
    """
    Get EC data path for a site
    """
    files = [
        f
        for f in os.listdir(path)
        if re.match(
            rf"AMF_{name}_FLUXNET(_FULLSET)?_(\d{{4}}-\d{{4}})_([A-Za-z0-9._-]+)\.zip",
            f,
        )
    ]
    if len(files) > 0:
        # Return first file
        return os.path.join(path, files[0])
    msg = f"File for site {name} not found"
    raise OSError(msg)


def get_best_fluxnet_column(df: pd.DataFrame, col: str):
    """
    Get best column to process fluxnet data
    """
    shift = col.count("_")
    if col in df.columns:
        return col
    if df.columns.str.startswith(f"{col}_").any():
        return min(
            (
                c
                for c in df.columns
                if c.startswith(f"{col}_") and "QC" not in c
            ),
            key=lambda c: c.split("_")[1 + shift],
        )
    return None


def process_fluxnet_column(
    df: pd.DataFrame, col: str, name: str | None = None
) -> pd.DataFrame:
    """
    Process a column of Fluxnet data
    """
    # LE
    best = get_best_fluxnet_column(df, col)
    if best is None:
        best = col
        df[col] = np.nan
    if name is None:
        name = col.lower()
    mapping = {best: name}
    if f"{best}_QC" in df.columns:
        mapping["best_QC"] = f"{name}_qc"
    return df.rename(columns=mapping)


def read_fluxnet_hh_data(name: str) -> pd.DataFrame:
    """
    Read EC data file for a site
    """
    # Read
    df = pd.read_csv(
        name,
        sep=",",
        encoding="ascii",
        usecols=(
            lambda col: (
                col
                in [
                    "TIMESTAMP_START",
                    "TIMESTAMP_END",
                    "LE_CORR",
                    "H_CORR",
                ]
                or col.startswith(
                    (
                        "G_F_MDS",
                        "TA_F_MDS",
                        "TS_F_MDS",
                        "RH",
                        "H_F_MDS",
                        "LE_F_MDS",
                        "NETRAD",
                        "SW_IN_F",
                        "SW_OUT",
                        "LW_IN_F",
                        "LW_OUT",
                    )
                )
            )
        ),
        parse_dates=["TIMESTAMP_START", "TIMESTAMP_END"],
    )
    # Rename columns
    df = df.rename(columns={"TIMESTAMP_START": "start", "TIMESTAMP_END": "end"})
    # print(df.columns)
    # LE
    df = process_fluxnet_column(df, "LE_F_MDS", "le")
    # LE CLOSED
    df = df.rename(columns={"LE_CORR": "le_closed"})
    # H
    df = process_fluxnet_column(df, "H_F_MDS", "h")
    # H CLOSED
    df = df.rename(columns={"H_CORR": "h_closed"})
    # RN
    df = process_fluxnet_column(df, "NETRAD", "rn")
    # SW IN
    df = process_fluxnet_column(df, "SW_IN_F", "sw_in")
    # SW OUT
    df = process_fluxnet_column(df, "SW_OUT", "sw_out")
    # LW IN
    df = process_fluxnet_column(df, "LW_IN_F", "lw_in")
    # LW OUT
    df = process_fluxnet_column(df, "LW_OUT", "lw_out")
    # G
    df = process_fluxnet_column(df, "G_F_MDS", "g")
    # TA
    df = process_fluxnet_column(df, "TA_F_MDS", "ta")
    # TS
    df = process_fluxnet_column(df, "TS_F_MDS", "ts")
    # RH
    df = process_fluxnet_column(df, "RH", "rh")
    # Keep required columns
    cols = [
        item
        for c in [
            "start",
            "end",
            "le",
            "h",
            "g",
            "rn",
            "sw_out",
            "sw_in",
            "lw_out",
            "lw_in",
            "ta",
            "ts",
            "rh",
            "le_closed",
            "h_closed",
        ]
        for item in ([c, f"{c}_qc"] if f"{c}_qc" in df.columns else [c])
    ]
    df = df[cols]
    # Exclude nodata
    value_cols = df.columns.difference(["start", "end"])
    # replace -9999 with NaN only in value columns
    df[value_cols] = df[value_cols].replace(-9999, np.nan)
    # drop rows where all value column is NaN
    df = df.dropna(subset=value_cols, how="all")
    with warnings.catch_warnings(action="ignore"):
        # Compute EF
        df["ef"] = df["le"] / (df["h"] + df["le"])
        # Compute ET
        df["et"] = df.apply(lambda x: compute_et_from_le(x["le"]), axis=1)
        # Compute solar net radiation
        df["srn"] = df["sw_in"] - df["sw_out"]
        # Compute dewpoint temperature
        eastar = 6.13753 * np.exp((17.27 * df["ta"]) / (df["ta"] + 237.3))
        ea = (df["rh"] / 100) * (eastar)
        df["td"] = 237.3 * np.log(ea / 6.13753) / (17.27 - np.log(ea / 6.13753))
        # Compute Rn - G
        df["rn-g"] = df["rn"] - df["g"]
    # Keep columns
    cols = [
        item
        for c in [
            "start",
            "end",
            "le",
            "h",
            "ef",
            "g",
            "rn",
            "srn",
            "sw_out",
            "sw_in",
            "lw_out",
            "lw_in",
            "ta",
            "ts",
            "td",
            "rh",
            "le_closed",
            "h_closed",
            "et",
            "rn-g",
        ]
        for item in ([c, f"{c}_qc"] if f"{c}_qc" in df.columns else [c])
    ]
    return df[cols]


def get_fluxnet_data(df: pd.DataFrame, path: str):
    """
    Get EC data at acquisition dates
    """
    # Get sites
    sites = list(df["name"].unique())
    # Loop for each site
    datasets = []
    ec_datasets = []
    errors = []
    for site in tqdm(sites, desc="Processing sites", unit="site"):
        try:
            # Select data for a site
            selected_df = df[df["name"] == site].reset_index(drop=True)
            # Code site
            code = get_code(site)
            # Read data
            archive = get_fluxnet_archive(code, path)
            with zipfile.ZipFile(archive, "r") as zip_ref:
                if "FULLSET" not in archive:
                    filename = f"{Path(archive).stem}.csv".replace(
                        "FLUXNET", "FLUXNET_FLUXMET_HH"
                    )
                else:
                    filename = f"{Path(archive).stem}.csv".replace(
                        "FLUXNET_FULLSET", "FLUXNET_FULLSET_HH"
                    )
                with zip_ref.open(filename) as file:
                    ec_df = read_fluxnet_hh_data(file)
                    ec_df = ec_df.rename(
                        columns={
                            v: f"fl_{v}"
                            for v in ec_df.columns
                            if v not in ["start", "end"]
                        }
                    )
            # Search indexes
            intervals = pd.IntervalIndex.from_arrays(
                ec_df["start"].dt.tz_localize(None),
                ec_df["end"].dt.tz_localize(None),
                closed="both",
            )
            intervals = intervals.astype("interval[datetime64[ns]]")
            matches, not_matches = intervals.get_indexer_non_unique(
                selected_df["date"].dt.tz_localize(None)
            )
            # Remove no matches
            matches = matches[matches != -1]
            datasets.append(
                selected_df.drop(not_matches).reset_index(drop=True)
            )
            ec_datasets.append(
                pd.concat(
                    [
                        selected_df[["name", "date"]]
                        .drop(not_matches)
                        .reset_index(drop=True),
                        ec_df.iloc[matches]
                        .drop(columns=["start", "end"])
                        .reset_index(drop=True),
                    ],
                    axis=1,
                )
            )
            tqdm.write(
                f"Site {site} processed with matches = {len(matches)} and not matches = {len(not_matches)}"
            )
        except Exception as e:  # noqa
            error_info = {
                "site": site,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            errors.append(error_info)
            tqdm.write(f"ERROR for site {site}: {e}")
    if len(errors) > 0:
        print("\n=== ERRORS ENCOUNTERED ===")
        for error in errors:
            print(f"\nSite: {error['site']}")
            print(f"Error: {error['error']}")
            print(error["traceback"])
    else:
        print("\nAll sites processed successfully!")

    return pd.concat(datasets), pd.concat(ec_datasets)

In [ ]:
# Functions for reading input data


def get_info(filename: str) -> pd.DataFrame:
    """
    Get information
    """
    return pd.read_excel(filename, sheet_name="INFO")


def read_input_data(filename: str) -> pd.DataFrame:
    """
    Read input data
    """
    sites = pd.ExcelFile(filename).sheet_names
    sites.remove("INFO")
    return pd.concat(
        [df for _, df in pd.read_excel(filename, sheet_name=sites).items()]
    )


def prepare_input_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data to run STIC
    """
    # ts, ta, td
    output_df = (
        df[["st_b10", "t2m", "d2m"]]
        .apply(convert_kelvin_to_celsius)
        .rename(columns={"st_b10": "ts", "t2m": "ta", "d2m": "td"})
    )
    # emis
    output_df["emis"] = df["st_emis"]
    # rh
    output_df["rh"] = output_df[["ta", "td"]].apply(
        lambda row: convert_to_rh(row["ta"], row["td"]), axis=1
    )
    # fc
    output_df["fc"] = df["fcover"]
    # lai
    output_df["lai"] = df["lai"]
    # ln
    eastar = output_df["ta"].apply(_tetens)
    ea = output_df["rh"] * eastar / 100
    output_df["ln"] = (
        df["st_emis"]
        * CST_SB
        * 1.24
        * (ea / (output_df["ta"] + KELVIN_CST)) ** (1 / 7)
        * (output_df["ta"] + KELVIN_CST) ** 4
        - df["st_emis"] * CST_SB * (output_df["ts"] + KELVIN_CST) ** 4
    )
    # rn
    output_df["rn"] = df["ssr"] + output_df["ln"]
    output_df["srn"] = df["ssr"]
    # localtime
    output_df["local_time"] = df[["Landsat_overpass", "Lon", "Lat"]].apply(
        lambda row: convert_to_local_time(
            date=dt.datetime.combine(
                row["Landsat_overpass"].date(),
                dt.time(hour=18, minute=45),
                tzinfo=dt.UTC,
            ),
            y=row["Lat"],
            x=row["Lon"],
            crs="epsg:4326",
        ),
        axis=1,
    )
    # name
    output_df["name"] = df["Name"]
    # lat/lon
    output_df["lon"] = df["Lon"]
    output_df["lat"] = df["Lat"]
    # Reflectance
    output_df["nir"] = df["sr_b5"]
    output_df["swir"] = df["sr_b6"]
    output_df["blue"] = df["sr_b2"]
    output_df["green"] = df["sr_b3"]
    output_df["red"] = df["sr_b4"]
    # Vegetation indices
    output_df["vari_green"] = utils.compute_vari_green(
        red=output_df["red"], green=output_df["green"], blue=output_df["blue"]
    )
    output_df["gli"] = utils.compute_gli(
        red=output_df["red"], green=output_df["green"], blue=output_df["blue"]
    )
    output_df["ndvi"] = utils.compute_ndvi(
        red=output_df["red"], nir=output_df["nir"]
    )
    output_df["gndvi"] = utils.compute_gndvi(
        nir=output_df["nir"], green=output_df["green"]
    )
    output_df["msavi"] = utils.compute_msavi(
        red=output_df["red"], nir=output_df["green"]
    )
    # dates
    output_df["date (utc)"] = df["Landsat_overpass"].apply(
        lambda t: dt.datetime.combine(
            t.date(), dt.time(hour=18, minute=45, tzinfo=dt.UTC)
        )
    )
    output_df["date"] = pd.to_datetime(
        df["Landsat_overpass"]
    ) + pd.to_timedelta(output_df["local_time"], unit="s")

    # reorder columns
    return output_df[
        [
            "name",
            "date",
            "date (utc)",
            "lat",
            "lon",
            "ts",
            "ta",
            "td",
            "rh",
            "fc",
            "lai",
            "srn",
            "rn",
            "ln",
            "local_time",
            "blue",
            "green",
            "red",
            "nir",
            "swir",
            "vari_green",
            "gli",
            "ndvi",
            "gndvi",
            "msavi",
            "emis",
        ]
    ]


def correct_data(
    df: pd.DataFrame,
    ec_df: pd.DataFrame,
    variables: list[str],
    dataset: str | None = None,
) -> pd.DataFrame:
    """
    Use EC data instead
    """
    # Get EC tower data
    if dataset is not None:
        ec_df = ec_df.rename(
            columns={
                v: f"ec_{v[len(dataset) + 1 :]}"
                for v in ec_df.columns
                if v not in ["name", "date"]
            }
        )
    ec_variables = [f"ec_{v}" for v in variables]
    ec_input = ec_df[["date", "name", *ec_variables]]
    corrected_df = pd.merge(left=df, right=ec_input, on=["date", "name"])
    return corrected_df.drop(columns=variables).rename(
        columns=dict(zip(ec_variables, variables, strict=True))
    )


def merge_data(
    df: pd.DataFrame,
    ec_df: pd.DataFrame,
    variables=list[str],
    dataset: str | None = None,
):
    cols = ["name", "date"] + [f"{dataset}_{v}" for v in variables]
    renamed_cols = {f"{dataset}_{v}": f"ec_{v}" for v in variables}
    return pd.merge(
        left=df,
        right=ec_df[cols].rename(columns=renamed_cols),
        on=["name", "date"],
    )

In [ ]:
# Functions for metrics


def compute_metrics(
    measured: npt.ArrayLike, estimated: npt.ArrayLike
) -> tuple[float, float, float, float, float]:
    """
    Compute slope, mbe, mae, rmse, r2
    """
    idx = np.isfinite(measured) & np.isfinite(estimated)
    slope, _ = np.polyfit(measured[idx], estimated[idx], 1)
    mbe = np.nanmean(estimated - measured)
    mae = mean_absolute_error(measured[idx], estimated[idx])
    rmse = np.sqrt(mean_squared_error(measured[idx], estimated[idx]))
    r2 = r2_score(measured[idx], estimated[idx])
    return (slope, mbe, mae, rmse, r2)


def search_data(df: pd.DataFrame, name: str, date: str) -> pd.DataFrame:
    """
    Search data
    """
    date_dt = dt.datetime.strptime(date, "%Y-%m-%d")
    query = f"name.str.contains('{name}') and date >= '{date_dt:%Y-%m-%d} 00:00:00' and date<'{date_dt + dt.timedelta(days=1):%Y-%m-%d} 00:00:00' "
    return df.query(query)


def plot_metrics(df: pd.DataFrame, variable: str = "le"):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {g: colors[i % len(colors)] for i, g in enumerate(base_groups)}

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            measured = df[f"ec_{variable}"]
            estimated = df[variable]
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[f"ec_{variable}"],
                        y=subset[variable],
                        mode="markers",
                        marker={
                            "size": 6,
                            "opacity": 0.5,
                            "color": color_map[g],
                        },
                        name=g,
                        legendgroup=g,
                        showlegend=(
                            group == "all"
                        ),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            "Measured: %{x:.3f}<br>"
                            "Estimated: %{y:.3f}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )
        else:
            subset = df[df.name == group]
            measured = subset[f"ec_{variable}"]
            estimated = subset[variable]

            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=measured,
                    y=estimated,
                    mode="markers",
                    marker={
                        "size": 6,
                        "opacity": 0.6,
                        "color": color_map.get(group, "gray"),
                    },
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        "Measured: %{x:.2f}<br>"
                        "Estimated: %{y:.2f}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

        # Identity line
        lims = [
            min(measured.min(), estimated.min()),
            max(measured.max(), estimated.max()),
        ]

        fig.add_trace(
            go.Scatter(
                x=lims,
                y=lims,
                mode="lines",
                line={"dash": "dash", "color": "black"},
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Linear fit
        idx = np.isfinite(measured) & np.isfinite(estimated)
        slope, intercept = np.polyfit(measured[idx], estimated[idx], 1)
        fig.add_trace(
            go.Scatter(
                x=lims,
                y=[lims[0] * slope + intercept, lims[1] * slope + intercept],
                mode="lines",
                line={"color": "red"},
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Metrics
        slope_m, mbe, mae, rmse, r2 = compute_metrics(measured, estimated)

        fig.add_annotation(
            x=0.97,
            y=0.03,
            xref="x domain",
            yref="y domain",
            text=(
                f"slope = {slope_m:.2f}<br>"
                f"MBE = {mbe:.2f}<br>"
                f"MAE = {mae:.2f}<br>"
                f"RMSE = {rmse:.2f}<br>"
                f"R2 = {r2:.2f}"
            ),
            showarrow=False,
            align="right",
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=f"Measured {variable.upper()}")
    fig.update_yaxes(title_text=f"Estimated {variable.upper()}")
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"Measured vs Estimated {variable.upper()}",
    )

    fig.show()

In [ ]:
# Functions for plotting


def plot_data(df: pd.DataFrame, variable: str):
    """
    Plot data
    """
    fig = px.line(
        df,
        x="date",
        y=variable,
        color="name",  # group by "name"
        markers=True,
        custom_data=["name"],
    )

    # Print value
    fig.update_traces(
        hovertemplate=(
            "<b>Site:</b> %{customdata[0]}<br>"
            "<b>Date:</b> %{x|%Y-%m-%d}<br>"
            f"<b>{variable}:</b> %{{y:.2f}}<br>"
            "<extra></extra>"
        )
    )

    # Format x-axis as YYYY-MM-DD
    fig.update_xaxes(tickformat="%Y-%m-%d")

    # Improve layout
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title=variable,
        legend_title="Site",
        title={
            "text": f"Evolution of {variable} over time",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 20},
        },
    )

    fig.show()


def plot_scatter(
    df: pd.DataFrame,
    x: str,
    y: str,
    axis_range: tuple[float, float] | None = None,
):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {g: colors[i % len(colors)] for i, g in enumerate(base_groups)}

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[x],
                        y=subset[y],
                        mode="markers",
                        marker={
                            "size": 6,
                            "opacity": 0.5,
                            "color": color_map[g],
                        },
                        name=g,
                        legendgroup=g,
                        showlegend=(
                            group == "all"
                        ),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            f"{x}: %{{x:.3f}}<br>"
                            f"{y}: %{{y:.3f}}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )
        else:
            subset = df[df.name == group]

            xdata = subset[x]
            ydata = subset[y]
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=xdata,
                    y=ydata,
                    mode="markers",
                    marker={
                        "size": 6,
                        "opacity": 0.6,
                        "color": color_map.get(group, "gray"),
                    },
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        f"{x}: %{{x:.3f}}<br>"
                        f"{y}: %{{y:.3f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

    fig.update_xaxes(title_text=f"{x}")
    fig.update_yaxes(title_text=f"{y}")
    if range is not None:
        fig.update_xaxes(range=axis_range)
        fig.update_yaxes(range=axis_range)
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"{x} vs {y}",
    )

    fig.show()


def plot_compare(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    x: str,
    y: str,
    axis_range: tuple[float, float] | None = None,
):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df1.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df1.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {g: colors[i % len(colors)] for i, g in enumerate(base_groups)}

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            for g in df1.name.unique():
                subset1 = df1[df1.name == g]
                subset2 = df2[df2.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset1[x],
                        y=subset2[y],
                        mode="markers",
                        marker={
                            "size": 6,
                            "opacity": 0.5,
                            "color": color_map[g],
                        },
                        name=g,
                        legendgroup=g,
                        showlegend=(
                            group == "all"
                        ),  # only show legend here if you want
                        customdata=subset1["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            f"{x}: %{{x:.3f}}<br>"
                            f"{y}: %{{y:.3f}}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )
        else:
            subset1 = df1[df1.name == group]
            subset2 = df2[df2.name == group]

            xdata = subset1[x]
            ydata = subset2[y]
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=xdata,
                    y=ydata,
                    mode="markers",
                    marker={
                        "size": 6,
                        "opacity": 0.6,
                        "color": color_map.get(group, "gray"),
                    },
                    name=group,
                    showlegend=False,
                    customdata=subset1["date"],
                    hovertemplate=(
                        f"{x}: %{{x:.3f}}<br>"
                        f"{y}: %{{y:.3f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

    fig.update_xaxes(title_text=f"{x}")
    fig.update_yaxes(title_text=f"{y}")
    if range is not None:
        fig.update_xaxes(range=axis_range)
        fig.update_yaxes(range=axis_range)
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"{x} vs {y}",
    )

    fig.show()

## Prepare input data

In [ ]:
stic_data_path = os.path.join(os.environ.get("SWEAT_TEST_DATA_PATH"), "stic")
input_data_path = os.path.join(
    stic_data_path, "California_2017-2025_PointScale.xlsx"
)

### Get information on input data

Input content

In [ ]:
info_df = get_info(input_data_path)

In [ ]:
info_df

### Read data

In [ ]:
input_data = read_input_data(input_data_path)

In [ ]:
input_data

### Prepare data

* Convert temperatures to Celsius degrees.
* Compute local time in seconds taking into account the Landsat's overpass over the area around 18h45 GMT, i.e. UTC.
* Compute long wave net radiation with the following formula :
$$
L_{net} = \epsilon * \sigma_{SB} * 1.24 * (ea/(T_a+273.15))^{1/7}*(T_a+273.15)^4 - \epsilon*\sigma_{SB}*(LST+273.15)^4
$$
* Compute net radiation from **ssr** and **lnet**

In [ ]:
data = prepare_input_data(input_data)

In [ ]:
data.head()

In [ ]:
data["nir/swir"] = data["nir"] / data["swir"]
data["ts-ta"] = data["ts"] - data["ta"]

### Plot data

In [ ]:
plot_data(data, variable="fc")

In [ ]:
plot_scatter(data, x="ta", y="ts")

### Use meteorological data and radiation data from EC tower

In [ ]:
_, am_data = get_ameriflux_data(data, stic_data_path)

In [ ]:
_, fl_data = get_fluxnet_data(data, stic_data_path)

In [ ]:
corrected_data = correct_data(
    data, am_data, variables=["ta", "td", "rh", "rn", "srn"], dataset="am"
)
corrected_data = merge_data(
    corrected_data,
    am_data,
    variables=["le", "le_closed", "h", "g", "rn-g"],
    dataset="am",
)
corrected_data.to_csv(
    os.path.join(stic_data_path, "data_landsat_ameriflux.csv"), index=False
)

In [ ]:
corrected_data = correct_data(
    data, fl_data, variables=["ta", "td", "rh", "rn", "srn"], dataset="fl"
)
corrected_data = merge_data(
    corrected_data,
    fl_data,
    variables=["le", "le_closed", "h", "g", "rn-g"],
    dataset="fl",
)
corrected_data.to_csv(
    os.path.join(stic_data_path, "data_landsat_fluxnet.csv"), index=False
)

## Read data

In [ ]:
data_path = os.path.join(stic_data_path, "data_landsat_ameriflux.csv")

In [ ]:
corrected_data = pd.read_csv(data_path, parse_dates=["date", "date (utc)"])

In [ ]:
corrected_data.head()

In [ ]:
plot_scatter(corrected_data, x="ta", y="ts", axis_range=(0, 60))

## Compute initialization of STIC model

In [ ]:
def init_stic(df: pd.DataFrame, version: str) -> pd.DataFrame:
    """
    Compute STIC model
    """
    res = run_batch_init_model(data=df, version=version)
    output_df = pd.DataFrame(
        {
            "name": df["name"].values,
            "date": df["date"],
            "le": res[:, 0],
            "h": res[:, 1],
            "g": res[:, 2],
            "rn": df["rn"].values,
            "ta": df["ta"].values,
            "td": df["td"].values,
            "rh": df["rh"].values,
            "ts": df["ts"].values,
            "ga": res[:, 3],
            "gs": res[:, 4],
            "t0": res[:, 5],
            "t0d": res[:, 6],
            "m": res[:, 7],
            "m_canopy": res[:, 8],
            "m_soil": res[:, 9],
            "m_surf": res[:, 10],
            "m_rz": res[:, 11],
            "da": res[:, 12],
            "ds": res[:, 13],
            "es": res[:, 14],
            "ea": res[:, 15],
            "e0": res[:, 16],
            "esstar": res[:, 17],
            "e0star": res[:, 18],
            "alpha": res[:, 19],
        }
    )
    cols = list(df.columns.difference(output_df.columns))
    cols.append("date")
    cols.append("name")
    return pd.merge(left=output_df, right=df[cols], on=["date", "name"])

In [ ]:
init_v1_3 = init_stic(corrected_data, version="1.3")

In [ ]:
init_v1_3

In [ ]:
plot_metrics(init_v1_3, variable="le")

In [ ]:
init_v1_4 = init_stic(corrected_data, version="1.4")

In [ ]:
plot_metrics(init_v1_4, variable="le")

## Compute STIC model

In [ ]:
def run_stic(df: pd.DataFrame, version: str) -> pd.DataFrame:
    """
    Compute STIC model
    """
    res = run_batch_model(data=df, threshold=0.05, nb_steps=15, version=version)
    output_df = pd.DataFrame(
        {
            "name": df["name"].values,
            "date": df["date"],
            "le": res[:, 0],
            "h": res[:, 1],
            "ef": res[:, 2],
            "g": res[:, 3],
            "rn": df["rn"].values,
            "ta": df["ta"].values,
            "td": df["td"].values,
            "rh": df["rh"].values,
            "ts": df["ts"].values,
            "ga": res[:, 4],
            "gs": res[:, 5],
            "t0": res[:, 6],
            "m": res[:, 7],
        }
    )
    output_df["et"] = output_df.apply(
        lambda x: compute_et_from_le(x["le"]), axis=1
    )
    output_df["rn-g"] = output_df["rn"] - output_df["g"]
    output_df["diff_le"] = output_df["le"] - df["ec_le"]
    output_df["diff_le_closed"] = output_df["le"] - df["ec_le_closed"]
    cols = list(df.columns.difference(output_df.columns))
    cols.append("date")
    cols.append("name")
    return pd.merge(left=output_df, right=df[cols], on=["date", "name"])

In [ ]:
res_v1_3 = run_stic(corrected_data, version="1.3")

In [ ]:
res_v1_3

In [ ]:
plot_metrics(res_v1_3, variable="le")

In [ ]:
res_v1_4 = run_stic(corrected_data, version="1.4")

In [ ]:
res_v1_4

In [ ]:
plot_metrics(res_v1_4, variable="le")

In [ ]:
plot_scatter(res_v1_4, x="ec_le_closed", y="le")

## Focus on one point

In [ ]:
def search(df: pd.DataFrame, name: str, date: str) -> pd.Series:
    """
    Search in dataframe
    """
    date_dt = dt.datetime.strptime(date, "%Y-%m-%d")
    query = f"name.str.contains('{name}') and date >= '{date_dt:%Y-%m-%d} 00:00:00' and date<'{date_dt + dt.timedelta(days=1):%Y-%m-%d} 00:00:00' "
    res = df.query(query)
    if len(res) > 1:
        msg = f"Too many results (Nb results found= {len(res)})"
        raise ValueError(msg)
    if len(res) == 0:
        msg = "Data not found"
        raise ValueError(msg)
    return res.iloc[0, :]


def focus_one_point(
    input_df: pd.DataFrame,
    name: str,
    date: str,
    version: str,
    results_df: pd.DataFrame | None = None,
):
    params = search(input_df, name, date)
    print(f"Date: {params['date']:%Y-%m-%d %H:%M:%S}")
    if results_df is not None:
        res = search(results_df, name, date)
        print(f"LE (estimated) = {res['le']:.3f}")
        print(f"LE (measured) = {res['ec_le']:.3f}")
    print("\nDetail STIC execution :")
    output = run_batch_model(
        pd.DataFrame([params]),
        threshold=0.01,
        nb_steps=15,
        debug=True,
        version=version,
    )
    print(f"LE (estimated) = {output[0, 0]:.3f}")
    print(f"H (estimated) = {output[0, 1]:.3f}")
    print(f"G (estimated) = {output[0, 3]:.3f}")
    print(f"ga (estimated) = {output[0, 4]:.3f}")
    print(f"gs (estimated) = {output[0, 5]:.3f}")

In [ ]:
search(corrected_data, name="Bi2", date="2021-04-06")

In [ ]:
focus_one_point(
    input_df=corrected_data,
    results_df=res_v1_3,
    name="Bi2",
    date="2020-06-22",
    version="1.3",
)

In [ ]:
focus_one_point(
    input_df=corrected_data,
    results_df=res_v1_4,
    name="Bi2",
    date="2020-06-22",
    version="1.4",
)